# Exponential and logarithm

This notebook covers `es.math.exp`, `log`, `log10`, `sqrt` and the error functions `erf` and `erfc`. They
take a `Parameter` or a `DescriptorNumber` and return a new one with the value, unit, variance and bounds
worked out: the variance to first order, the bounds as the exact range of the function over the bounds of
the input.

The examples are taken from a diffraction experiment, but each one stands alone. Each part closes with a few exercises on what was just shown. Their hints and solutions are folded away and open on a click, so the text reads straight through; a solution is a worked answer with its result.

In [ ]:
import numpy as np
import plotly.graph_objects as go

import easyscience as es
from easyscience import DescriptorNumber
from easyscience import Parameter

The function `show` prints a number as value, error and unit, and, for a `Parameter` with finite bounds, the bounds.

In [ ]:
def show(number):
    """One line with value, error, unit and, for a Parameter, the finite bounds."""
    text = f'{number.value:.4f} ± {number.error or 0.0:.4f} {number.unit}'
    if hasattr(number, 'min') and (np.isfinite(number.min) or np.isfinite(number.max)):
        text += f'   in [{number.min:.4f}, {number.max:.4f}]'
    return text

In [ ]:
import base64
import contextlib
import html
import io
import re

from IPython import get_ipython
from IPython.display import HTML
from IPython.display import IFrame
from IPython.display import display
from pygments import highlight
from pygments.formatters import HtmlFormatter
from pygments.lexers import PythonLexer


def show_figure(fig: go.Figure, height: int = 420) -> None:
    """Show a Plotly figure as an embedded iframe, so that it renders in the docs."""
    page = (
        go
        .Figure(fig)
        .update_layout(template='plotly_white')
        .to_html(
            full_html=True,
            include_plotlyjs='cdn',
            default_width='100%',
            default_height=f'{height - 20}px',
        )
    )
    encoded = base64.b64encode(page.encode('utf-8')).decode('ascii')
    display(IFrame(src=f'data:text/html;base64,{encoded}', width='100%', height=height))


def solution(note: str, source: str) -> None:
    """Run a solution in a copy of the notebook namespace and show it, folded, with its output."""
    code = source.strip('\n')
    namespace = dict(get_ipython().user_ns)
    stream = io.StringIO()
    with contextlib.redirect_stdout(stream):
        exec(code, namespace)
    parts = [
        '<details class="example"><summary>Solution</summary>',
        highlight(code, PythonLexer(), HtmlFormatter()),
    ]
    result = stream.getvalue().rstrip('\n')
    if result:
        parts.append('<p>Output:</p><pre>' + html.escape(result) + '</pre>')
    if note.strip():
        text = re.sub('`([^`]+)`', r'<code>\1</code>', html.escape(note.strip()))
        parts.append('<p>' + text + '</p>')
    parts.append('</details>')
    display(HTML(''.join(parts)))


get_ipython().register_magic_function(solution, magic_kind='cell')

## The exponential

`exp` takes a dimensionless number. The Debye-Waller factor of a crystal, $\exp\!\left(-q^2\langle
u^2\rangle/3\right)$, combines a scattering vector $q$ and a mean-square displacement $\langle u^2\rangle$;
the units cancel, so `exp` accepts it:

In [ ]:
q = Parameter(name='q', value=2.0, unit='1/angstrom', variance=0.02**2, min=0.5, max=4.0)
u2 = Parameter(name='u2', value=0.01, unit='angstrom**2', variance=0.002**2, min=0.0, max=0.05)

debye_waller = es.math.exp(-(q**2) * u2 / 3)
print(show(debye_waller))

The bounds are the exact range of the function over the bounds of the inputs: the factor can be as small as
0.77 and cannot exceed 1.

### Exercise 1: Transmission through a sample

The transmission of a beam through a sample of thickness $t$ is $T = \exp(-\mu t)$. For $\mu = 0.50 \pm 0.02$ 1/cm and $t = 2.0 \pm 0.1$ cm, what is $T$, with its uncertainty?

In [ ]:
mu = Parameter(name='mu', value=0.50, variance=0.02**2, unit='1/cm')
t = Parameter(name='t', value=2.0, variance=0.1**2, unit='cm')

<details class="question">
<summary>Hint 1</summary>

The product of `mu` and `t` has no unit: `1/cm` times `cm` cancels, which is why `exp` accepts it.

</details>

<details class="question">
<summary>Hint 2</summary>

Write the argument first, `-mu * t`, and pass it to `es.math.exp`.

</details>

In [ ]:
%%solution The lower bound 0 appears because an exponential is never negative.
transmission = es.math.exp(-mu * t)
print(show(transmission))

## Logarithms

`log` is the natural logarithm and `log10` the decimal one. Both need a dimensionless, positive number. A
relative intensity in decibels uses `log10`; here the reference is a `DescriptorNumber` without an
uncertainty:

In [ ]:
intensity = Parameter(name='intensity', value=250.0, variance=10.0**2, min=100.0, max=400.0)
reference = DescriptorNumber(name='reference', value=1000.0, variance=0.0)

decibel = 10 * es.math.log10(intensity / reference)
print(show(decibel))

A value outside the domain is refused with an error that names the parameter:

In [ ]:
try:
    es.math.log10(Parameter(name='v', value=0.0, variance=0.1**2))
except ValueError as error:
    print(error)

### Exercise 2: Decibels

A detector counts $I = 250 \pm 10$ in a run where the reference intensity is $I_0 = 1000$, known exactly. What is the level $10 \log_{10}(I/I_0)$ in decibels?

In [ ]:
intensity = Parameter(name='intensity', value=250.0, variance=10.0**2)
reference = DescriptorNumber(name='reference', value=1000.0, variance=0.0)

<details class="question">
<summary>Hint 1</summary>

Divide the two intensities first.

</details>

<details class="question">
<summary>Hint 2</summary>

`log10` takes a dimensionless number; the factor 10 is a plain number that multiplies the result.

</details>

In [ ]:
%%solution
level = 10 * es.math.log10(intensity / reference)
print(show(level))

## The square root

`sqrt` also accepts a number with a unit, as long as the unit can be halved. The amplitude of a vector from
its two components:

In [ ]:
a = Parameter(name='a', value=3.0, variance=0.1**2, min=2.0, max=4.0)
b = Parameter(name='b', value=4.0, variance=0.1**2, min=3.0, max=5.0)

amplitude = es.math.sqrt(a**2 + b**2)
print(show(amplitude))

The square root of an area is a length. The square root of a length is not supported by the unit library, and is
refused:

In [ ]:
area = Parameter(name='area', value=9.0, unit='m**2', variance=0.3**2, min=4.0, max=16.0)
print(show(es.math.sqrt(area)))

try:
    es.math.sqrt(Parameter(name='v', value=2.0, unit='m', variance=0.1**2))
except Exception as error:
    print(f'{type(error).__name__}: {error}')

### Exercise 3: Amplitude from intensity

The intensity of a diffraction peak, in units of a standard, is $0.64 \pm 0.02$. Intensity is proportional to the square of the amplitude. What is the amplitude, with its uncertainty?

In [ ]:
intensity = Parameter(name='intensity', value=0.64, variance=0.02**2)

<details class="question">
<summary>Hint 1</summary>

The amplitude is the square root of the intensity.

</details>

<details class="question">
<summary>Hint 2</summary>

The relative error of a square root is half the relative error of its argument.

</details>

In [ ]:
%%solution The lower bound 0 is the lower end of the domain of the square root.
amplitude = es.math.sqrt(intensity)
print(show(amplitude))

## The error functions

The fraction of a Gaussian beam of width $\sigma$ that passes a slit of half-width $a$ is
$\mathrm{erf}\!\left(a/(\sigma\sqrt{2})\right)$, and `erfc` $= 1 - \mathrm{erf}$ is the fraction that is
blocked:

In [ ]:
slit = Parameter(name='slit', value=2.0, unit='mm', variance=0.1**2, min=1.0, max=3.0)
beam = Parameter(name='beam', value=1.5, unit='mm', variance=0.05**2, min=1.0, max=2.0)

transmitted = es.math.erf(slit / (beam * np.sqrt(2)))
blocked = es.math.erfc(slit / (beam * np.sqrt(2)))
print(show(transmitted))
print(show(blocked))
print(transmitted.value + blocked.value)

### Exercise 4: Slit transmission

A Gaussian beam with $\sigma = 0.80 \pm 0.04$ mm hits a slit with a half-width of $1.0 \pm 0.05$ mm. What fraction of the beam passes?

In [ ]:
half_width = Parameter(name='half_width', value=1.0, variance=0.05**2, unit='mm')
beam_sigma = Parameter(name='beam_sigma', value=0.8, variance=0.04**2, unit='mm')

<details class="question">
<summary>Hint 1</summary>

The fraction is $\mathrm{erf}(a/(\sigma\sqrt{2}))$. The two lengths are in the same unit, so their ratio has no unit.

</details>

<details class="question">
<summary>Hint 2</summary>

`np.sqrt(2)` is a plain number; divide by `beam_sigma * np.sqrt(2)`.

</details>

In [ ]:
%%solution With unbounded inputs the bounds of `erf` are its range, from -1 to 1.
fraction = es.math.erf(half_width / (beam_sigma * np.sqrt(2)))
print(show(fraction))

## A round trip returns where it started

The derivative of $\exp(\log x)$ is 1, so the value **and** the uncertainty come back unchanged:

In [ ]:
x = Parameter(name='x', value=1.2, variance=0.05**2, min=0.5, max=3.0)
round_trip = es.math.exp(es.math.log(x))
print(show(round_trip))
print(np.isclose(round_trip.variance, x.variance))

## Charts

The Debye-Waller factor falls off with $q$, and its uncertainty comes from the uncertainty of
$\langle u^2\rangle = 0.010 \pm 0.002$ Å$^2$. The chart is interactive: hover over the curve to read the
values. It loads Plotly from the internet, so it needs a connection. The charts are shown with a small
helper, `show_figure(fig)`, which is defined in a hidden cell; in your own notebook `fig.show()` does the
same.

In [ ]:
q_values = np.linspace(0.2, 4.0, 40)
u2 = Parameter(name='u2', value=0.01, unit='angstrom**2', variance=0.002**2)

values, errors = [], []
for value in q_values:
    q = DescriptorNumber(name='q', value=value, unit='1/angstrom', variance=0.0)
    factor = es.math.exp(-(q**2) * u2 / 3)
    values.append(factor.value)
    errors.append(factor.error)
values, errors = np.array(values), np.array(errors)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=q_values,
        y=values - errors,
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=q_values,
        y=values + errors,
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(99, 110, 250, 0.25)',
        name='± 1 error',
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=q_values,
        y=values,
        mode='lines',
        name='Debye-Waller factor',
        line=dict(color='rgb(99, 110, 250)'),
        customdata=errors,
        hovertemplate='q = %{x:.2f} 1/Å<br>factor = %{y:.4f} ± %{customdata:.4f}<extra></extra>',
    )
)
fig.update_layout(
    xaxis_title='q / (1/Å)',
    yaxis_title='Debye-Waller factor',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.55, y=0.95),
)
show_figure(fig)

The fraction of a Gaussian beam that passes a slit grows with the half-width $a$ and saturates at 1. The
beam width is $0.80 \pm 0.04$ mm, and the uncertainty of the fraction comes from that width: it is largest when the half-width is close to the beam width itself, here around 0.8 mm:

In [ ]:
half_widths = np.linspace(0.1, 3.0, 59)
beam = Parameter(name='beam', value=0.8, unit='mm', variance=0.04**2)

values, errors = [], []
for value in half_widths:
    half_width = DescriptorNumber(name='half_width', value=value, unit='mm', variance=0.0)
    fraction = es.math.erf(half_width / (beam * np.sqrt(2)))
    values.append(fraction.value)
    errors.append(fraction.error)
values, errors = np.array(values), np.array(errors)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=half_widths,
        y=values - errors,
        mode='lines',
        line=dict(width=0),
        showlegend=False,
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=half_widths,
        y=values + errors,
        mode='lines',
        line=dict(width=0),
        fill='tonexty',
        fillcolor='rgba(99, 110, 250, 0.25)',
        name='± 1 error',
        hoverinfo='skip',
    )
)
fig.add_trace(
    go.Scatter(
        x=half_widths,
        y=values,
        mode='lines',
        name='fraction that passes',
        line=dict(color='rgb(99, 110, 250)'),
        customdata=errors,
        hovertemplate='a = %{x:.2f} mm<br>fraction = %{y:.4f} ± %{customdata:.4f}<extra></extra>',
    )
)
fig.update_layout(
    xaxis_title='slit half-width a / mm',
    yaxis_title='fraction that passes',
    margin=dict(l=60, r=20, t=30, b=50),
    legend=dict(x=0.55, y=0.4),
)
show_figure(fig, height=380)

## Good to know

The first-order variance ignores the curvature of the function, so it is accurate only while the
uncertainty of the argument is small compared with the distance over which the slope changes. For $x \sim \mathcal{N}(1, 1)$ the first-order variance of $\exp(x)$ is $e^2 = 7.389$, but the
exact variance of $\exp(x)$ is $e^3(e - 1) = 34.51$:

In [ ]:
wide = Parameter(name='wide', value=1.0, variance=1.0**2)
print(es.math.exp(wide).variance)
print(np.exp(3) * (np.e - 1))

With an uncertainty of 0.05 in the argument the first-order variance of $\exp(x)$ is within half a per cent of the exact one.